# Deep Past Challenge: Akkadian to English Translation
## Kaggle Competition - Machine Translation for Ancient Languages

---

### Competition Overview
Translating **4,000-year-old Assyrian merchant tablets** from transliterated Akkadian to English.

**Key Challenge**: Low-resource, morphologically complex language where a single word can encode subject, object, tense, and more.

### Evaluation Metric
```
Score = sqrt(BLEU x chrF++)
```

### Strategy
1. **Data Preprocessing** - Clean transliterations using competition guidelines
2. **Data Augmentation** - Extract additional training data from publications
3. **Model Fine-tuning** - Use pre-trained Akkadian-specific models
4. **Ensemble** - Combine multiple models for robust predictions

In [1]:
# ============================================================================
# IMPORTS & CONFIGURATION
# ============================================================================

import os
import re
import gc
import warnings
import unicodedata
from pathlib import Path
from typing import List, Dict, Tuple, Optional

# IMPORTANT: Disable TensorFlow before importing transformers
# This prevents DLL loading issues on Windows
os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_NO_TF"] = "1"

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Deep Learning (PyTorch only)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Hugging Face (PyTorch backend)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import Dataset as HFDataset

# Evaluation
import sacrebleu
from sacrebleu.metrics import BLEU, CHRF

# Aliases for compatibility
T5Tokenizer = AutoTokenizer
T5ForConditionalGeneration = AutoModelForSeq2SeqLM

# Suppress warnings
warnings.filterwarnings('ignore')

# Configuration
class Config:
    # Paths - Adjust for Kaggle environment
    KAGGLE_INPUT = Path("/kaggle/input/deep-past-initiative-machine-translation")
    KAGGLE_MODELS = Path("/kaggle/input")
    OUTPUT_DIR = Path("/kaggle/working")
    
    # Local paths for development
    LOCAL_INPUT = Path(".")
    
    # Model settings
    MODEL_NAME = "t5-base"  # Base model for fine-tuning
    MAX_SOURCE_LENGTH = 256
    MAX_TARGET_LENGTH = 256
    
    # Training settings
    BATCH_SIZE = 8
    GRADIENT_ACCUMULATION = 4
    LEARNING_RATE = 3e-5
    EPOCHS = 5
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.01
    
    # Inference
    NUM_BEAMS = 5
    LENGTH_PENALTY = 1.0
    REPETITION_PENALTY = 1.2
    
    # Device
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Random seed
    SEED = 42

# Set seeds for reproducibility
def set_seed(seed: int = Config.SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

print(f"[CONFIG] Device: {Config.DEVICE}")
print(f"[CONFIG] PyTorch version: {torch.__version__}")
print(f"[CONFIG] CUDA available: {torch.cuda.is_available()}")


c:\Users\carlo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[CONFIG] Device: cuda
[CONFIG] PyTorch version: 2.5.1+cu121
[CONFIG] CUDA available: True


## 1. Data Loading & Exploration


In [2]:
# ============================================================================
# DATA LOADING
# ============================================================================

def get_data_path():
    """Get the appropriate data path for Kaggle or local environment."""
    if Config.KAGGLE_INPUT.exists():
        return Config.KAGGLE_INPUT
    return Config.LOCAL_INPUT

DATA_PATH = get_data_path()
print(f"[INFO] Data path: {DATA_PATH}")

# Load main datasets
train_df = pd.read_csv(DATA_PATH / "train.csv")
test_df = pd.read_csv(DATA_PATH / "test.csv")

print(f"\n[INFO] Training set shape: {train_df.shape}")
print(f"[INFO] Test set shape: {test_df.shape}")

# Display sample data
print("\n" + "="*60)
print("TRAINING DATA SAMPLE")
print("="*60)
display(train_df.head(3))

print("\n" + "="*60)
print("TEST DATA SAMPLE")
print("="*60)
display(test_df.head(3))


[INFO] Data path: .

[INFO] Training set shape: (1561, 3)
[INFO] Test set shape: (4, 5)

TRAINING DATA SAMPLE


,oare_id,transliteration,translation
0,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ..."
1,0064939c-59b9-4448-a63d-34612af0a1b5,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...
2,0073f2c0-524c-4bbf-915a-8c1772a4fb98,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,... he did not give you a textile. He returned...



TEST DATA SAMPLE


,id,text_id,line_start,line_end,transliteration
0,0,332fda50,1,7,um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-t...
1,1,332fda50,7,14,i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-n...
2,2,332fda50,14,24,ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na...


In [3]:
# ============================================================================
# DATA EXPLORATION
# ============================================================================

print("TRAINING DATA STATISTICS")
print("="*60)

# Text length analysis
train_df['transliteration_len'] = train_df['transliteration'].apply(lambda x: len(str(x).split()))
train_df['translation_len'] = train_df['translation'].apply(lambda x: len(str(x).split()))

print(f"\nTransliteration length (words):")
print(f"   Mean: {train_df['transliteration_len'].mean():.1f}")
print(f"   Median: {train_df['transliteration_len'].median():.1f}")
print(f"   Max: {train_df['transliteration_len'].max()}")
print(f"   Min: {train_df['transliteration_len'].min()}")

print(f"\nTranslation length (words):")
print(f"   Mean: {train_df['translation_len'].mean():.1f}")
print(f"   Median: {train_df['translation_len'].median():.1f}")
print(f"   Max: {train_df['translation_len'].max()}")
print(f"   Min: {train_df['translation_len'].min()}")

# Missing values
print(f"\nMissing values in train:")
print(train_df.isnull().sum())

# Sample transliteration-translation pairs
print("\n" + "="*60)
print("SAMPLE TRANSLITERATION-TRANSLATION PAIRS")
print("="*60)

for i in range(min(3, len(train_df))):
    print(f"\n--- Example {i+1} ---")
    print(f"Akkadian: {train_df.iloc[i]['transliteration'][:200]}...")
    print(f"English:  {train_df.iloc[i]['translation'][:200]}...")


TRAINING DATA STATISTICS

Transliteration length (words):
   Mean: 57.5
   Median: 49.0
   Max: 187
   Min: 3

Translation length (words):
   Mean: 90.5
   Median: 68.0
   Max: 744
   Min: 1

Missing values in train:
oare_id                0
transliteration        0
translation            0
transliteration_len    0
translation_len        0
dtype: int64

SAMPLE TRANSLITERATION-TRANSLATION PAIRS

--- Example 1 ---
Akkadian: KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠIB šu-(d)EN.LÍL DUMU ma-nu-ki-a-šur KIŠIB MAN-a-šur DUMU a-ta-a 0.33333 ma-na 2 GÍN KÙ.BABBAR SIG₅ i-ṣé-er PUZUR₄-a-šur DUMU a-ta-a a-lá-ḫu-um i-šu iš-tù ḫ...
English:  Seal of Mannum-balum-Aššur son of Ṣilli-Adad, seal of Šu-Illil son of Mannum-kī-Aššur, seal of Puzur-Aššur son of Ataya. Puzur-Aššur son of Ataya owes 22 shekels of good silver to Ali-ahum. Reckoned f...

--- Example 2 ---
Akkadian: 1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé...
English:  Itūr-ilī has received one textile of ordinary quality....

--- Example 3 ---


## 2. Text Preprocessing

Following the competition's **Dataset Instructions** for formatting Akkadian transliterations:
- Remove modern scribal notations: `!`, `?`, `/`, `:`, `.`
- Handle determinatives in curly braces: `{d}`, `{ki}`, `{m}`, etc.
- Standardize breaks and gaps: `[x]` -> `<gap>`, `[...]` -> `<big_gap>`
- Convert subscript numbers: `il5` -> `il5`
- Remove partial bracket notations: half-brackets, `<`, `>`, `[`, `]`


In [4]:
# ============================================================================
# TEXT PREPROCESSING FOR AKKADIAN
# ============================================================================

class AkkadianPreprocessor:
    """
    Preprocessor for Akkadian transliterations following competition guidelines.
    """
    
    # Subscript to regular number mapping
    SUBSCRIPT_MAP = {
        '₀': '0', '₁': '1', '₂': '2', '₃': '3', '₄': '4',
        '₅': '5', '₆': '6', '₇': '7', '₈': '8', '₉': '9',
        '⁰': '0', '¹': '1', '²': '2', '³': '3', '⁴': '4',
        '⁵': '5', '⁶': '6', '⁷': '7', '⁸': '8', '⁹': '9'
    }
    
    # Determinatives to preserve (in simplified form)
    DETERMINATIVES = [
        '{d}', '{ki}', '{m}', '{f}', '{mi}', '{lu₂}', '{lu2}',
        '{uru}', '{kur}', '{e₂}', '{e2}', '{geš}', '{ĝeš}',
        '{tug₂}', '{tug2}', '{dub}', '{mul}', '{id₂}', '{id2}',
        '{mušen}', '{na₄}', '{na4}', '{kuš}', '{u₂}', '{u2}'
    ]
    
    @staticmethod
    def convert_subscripts(text: str) -> str:
        """Convert subscript/superscript numbers to regular numbers."""
        for sub, num in AkkadianPreprocessor.SUBSCRIPT_MAP.items():
            text = text.replace(sub, num)
        return text
    
    @staticmethod
    def normalize_determinatives(text: str) -> str:
        """Normalize determinatives (keep them but standardize format)."""
        # Convert common variations
        text = re.sub(r'\{lu₂\}', '{lu2}', text)
        text = re.sub(r'\{e₂\}', '{e2}', text)
        text = re.sub(r'\{tug₂\}', '{tug2}', text)
        text = re.sub(r'\{id₂\}', '{id2}', text)
        text = re.sub(r'\{na₄\}', '{na4}', text)
        text = re.sub(r'\{u₂\}', '{u2}', text)
        text = re.sub(r'\{ĝeš\}', '{ges}', text)
        return text
    
    @staticmethod
    def remove_scribal_notations(text: str) -> str:
        """Remove modern scribal notations."""
        # Remove certain reading markers
        text = re.sub(r'!', '', text)
        text = re.sub(r'\?', '', text)
        
        # Remove line dividers (but keep in context)
        text = re.sub(r'\s*/\s*', ' ', text)
        
        # Remove word dividers
        text = re.sub(r'\s*:\s*', ' ', text)
        text = re.sub(r'\s*\.\s*', ' ', text)
        
        return text
    
    @staticmethod
    def handle_breaks(text: str) -> str:
        """Standardize break and gap markers."""
        # Large gaps
        text = re.sub(r'\[…+\s*…*\]', '<big_gap>', text)
        text = re.sub(r'…+', '<big_gap>', text)
        text = re.sub(r'\[\.\.\.\]', '<big_gap>', text)
        
        # Small gaps
        text = re.sub(r'\[x+\]', '<gap>', text)
        text = re.sub(r'\[X+\]', '<gap>', text)
        
        return text
    
    @staticmethod
    def remove_brackets(text: str) -> str:
        """Remove various bracket notations but keep content."""
        # Half brackets (partially broken signs)
        text = re.sub(r'[˹˺]', '', text)
        
        # Scribal insertions - keep text but remove brackets
        text = re.sub(r'<([^>]*)>', r'\1', text)
        
        # Square brackets - keep text but remove brackets
        text = re.sub(r'\[([^\]]*)\]', r'\1', text)
        
        # Double pointy brackets (erroneous signs) - remove entirely
        text = re.sub(r'<<[^>]*>>', '', text)
        
        # Parentheses with comments
        text = re.sub(r'\([^)]*\)', '', text)
        
        return text
    
    @staticmethod
    def normalize_whitespace(text: str) -> str:
        """Normalize whitespace."""
        text = re.sub(r'\s+', ' ', text)
        return text.strip()
    
    @classmethod
    def preprocess(cls, text: str, keep_determinatives: bool = True) -> str:
        """
        Full preprocessing pipeline for Akkadian transliterations.
        """
        if pd.isna(text) or not text:
            return ""
        
        text = str(text)
        
        # Apply preprocessing steps
        text = cls.convert_subscripts(text)
        text = cls.remove_scribal_notations(text)
        text = cls.handle_breaks(text)
        text = cls.remove_brackets(text)
        
        if keep_determinatives:
            text = cls.normalize_determinatives(text)
        else:
            # Remove all determinatives
            text = re.sub(r'\{[^}]*\}', '', text)
        
        text = cls.normalize_whitespace(text)
        
        return text
    
    @classmethod
    def preprocess_translation(cls, text: str) -> str:
        """
        Light preprocessing for English translations.
        """
        if pd.isna(text) or not text:
            return ""
        
        text = str(text)
        text = cls.normalize_whitespace(text)
        
        return text

# Test preprocessing
print("PREPROCESSING TEST")
print("="*60)

sample_text = "um-ma A-šur-{ki} ša-ru-u2! {m}Ša-lim-a-hu-um [x x] i-na? KÙ.BABBAR"
print(f"Original:    {sample_text}")
print(f"Preprocessed: {AkkadianPreprocessor.preprocess(sample_text)}")


PREPROCESSING TEST
Original:    um-ma A-šur-{ki} ša-ru-u2! {m}Ša-lim-a-hu-um [x x] i-na? KÙ.BABBAR
Preprocessed: um-ma A-šur-{ki} ša-ru-u2 {m}Ša-lim-a-hu-um x x i-na KÙ BABBAR


In [5]:
# ============================================================================
# APPLY PREPROCESSING TO DATASETS
# ============================================================================

print("Applying preprocessing to datasets...")

# Preprocess training data
train_df['transliteration_clean'] = train_df['transliteration'].apply(
    AkkadianPreprocessor.preprocess
)
train_df['translation_clean'] = train_df['translation'].apply(
    AkkadianPreprocessor.preprocess_translation
)

# Preprocess test data
test_df['transliteration_clean'] = test_df['transliteration'].apply(
    AkkadianPreprocessor.preprocess
)

# Remove empty rows
initial_train_size = len(train_df)
train_df = train_df[
    (train_df['transliteration_clean'].str.len() > 0) & 
    (train_df['translation_clean'].str.len() > 0)
].reset_index(drop=True)

print(f"[OK] Training samples after cleaning: {len(train_df)} (removed {initial_train_size - len(train_df)})")
print(f"[OK] Test samples: {len(test_df)}")

# Show preprocessed samples
print("\n" + "="*60)
print("PREPROCESSED SAMPLES")
print("="*60)

for i in range(min(2, len(train_df))):
    print(f"\n--- Example {i+1} ---")
    print(f"Original:     {train_df.iloc[i]['transliteration'][:100]}...")
    print(f"Preprocessed: {train_df.iloc[i]['transliteration_clean'][:100]}...")
    print(f"Translation:  {train_df.iloc[i]['translation_clean'][:100]}...")


Applying preprocessing to datasets...
[OK] Training samples after cleaning: 1561 (removed 0)
[OK] Test samples: 4

PREPROCESSED SAMPLES

--- Example 1 ---
Original:     KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠIB šu-(d)EN.LÍL DUMU ma-nu-ki-a-šur KIŠIB MAN-a-šur DUM...
Preprocessed: KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-IM KIŠIB šu-EN LÍL DUMU ma-nu-ki-a-šur KIŠIB MAN-a-šur DUMU a-ta...
Translation:  Seal of Mannum-balum-Aššur son of Ṣilli-Adad, seal of Šu-Illil son of Mannum-kī-Aššur, seal of Puzur...

--- Example 2 ---
Original:     1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé...
Preprocessed: 1 TÚG ša qá-tim i-tur4-DINGIR il5-qé...
Translation:  Itūr-ilī has received one textile of ordinary quality....


## 3. Load Supplementary Data & Lexicon

Loading additional resources for enhanced translation.


In [6]:
# ============================================================================
# LOAD SUPPLEMENTARY DATA
# ============================================================================

# Load lexicon for vocabulary enrichment
try:
    lexicon_df = pd.read_csv(DATA_PATH / "OA_Lexicon_eBL.csv")
    print(f"[INFO] Lexicon loaded: {len(lexicon_df)} entries")
    display(lexicon_df.head(3))
except Exception as e:
    print(f"[WARN] Could not load lexicon: {e}")
    lexicon_df = None

# Load dictionary
try:
    dictionary_df = pd.read_csv(DATA_PATH / "eBL_Dictionary.csv")
    print(f"\n[INFO] Dictionary loaded: {len(dictionary_df)} entries")
    display(dictionary_df.head(3))
except Exception as e:
    print(f"[WARN] Could not load dictionary: {e}")
    dictionary_df = None

# Load published texts for additional context
try:
    published_df = pd.read_csv(DATA_PATH / "published_texts.csv")
    print(f"\n[INFO] Published texts loaded: {len(published_df)} texts")
    print(f"   Columns: {published_df.columns.tolist()}")
except Exception as e:
    print(f"[WARN] Could not load published texts: {e}")
    published_df = None


[INFO] Lexicon loaded: 39332 entries


,type,form,norm,lexeme,eBL,I_IV,A_D,Female(f),Alt_lex
0,word,áb ša-ra-ni,ab šarrānē,ab šarrānē,https://www.ebl.lmu.de/dictionary?word=ab šarrānē,NaN,NaN,NaN,NaN
1,word,áb ša-ra-nim,ab šarrānem,ab šarrānē,https://www.ebl.lmu.de/dictionary?word=ab šarrānē,NaN,NaN,NaN,NaN
2,word,áb-ša-ra-nim,ab šarrānem,ab šarrānē,https://www.ebl.lmu.de/dictionary?word=ab šarrānē,NaN,NaN,NaN,NaN



[INFO] Dictionary loaded: 19215 entries


,word,definition,derived_from
0,-a I,"""my"" (1 sg. pron. suff.)",cf. -ī I
1,-am I,"""to me"" (1 sg. dat. suff.) 2. vent. affix (cf....",NaN
2,-atti I,(adv. endings) (cf. GAG §113l),NaN



[INFO] Published texts loaded: 7953 texts
   Columns: ['oare_id', 'online transcript', 'cdli_id', 'aliases', 'label', 'publication_catalog', 'description', 'genre_label', 'inventory_position', 'online_catalog', 'note', 'interlinear_commentary', 'online_information', 'excavation_no', 'oatp_key', 'eBL_id', 'AICC_translation', 'transliteration_orig', 'transliteration']


## 4. Model Setup & Dataset Preparation


In [7]:
# ============================================================================
# DATASET CLASS FOR TRANSLATION
# ============================================================================

class AkkadianTranslationDataset(Dataset):
    """PyTorch Dataset for Akkadian to English translation."""
    
    def __init__(
        self, 
        transliterations: List[str], 
        translations: Optional[List[str]] = None,
        tokenizer = None,
        max_source_length: int = Config.MAX_SOURCE_LENGTH,
        max_target_length: int = Config.MAX_TARGET_LENGTH,
        prefix: str = "translate Akkadian to English: "
    ):
        self.transliterations = transliterations
        self.translations = translations
        self.tokenizer = tokenizer
        self.max_source_length = max_source_length
        self.max_target_length = max_target_length
        self.prefix = prefix
    
    def __len__(self):
        return len(self.transliterations)
    
    def __getitem__(self, idx):
        source = self.prefix + self.transliterations[idx]
        
        # Tokenize source
        source_encoding = self.tokenizer(
            source,
            max_length=self.max_source_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        item = {
            "input_ids": source_encoding["input_ids"].squeeze(),
            "attention_mask": source_encoding["attention_mask"].squeeze(),
        }
        
        # Tokenize target if available (training mode)
        if self.translations is not None:
            target_encoding = self.tokenizer(
                self.translations[idx],
                max_length=self.max_target_length,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )
            
            labels = target_encoding["input_ids"].squeeze()
            # Replace padding token id with -100 for loss calculation
            labels[labels == self.tokenizer.pad_token_id] = -100
            item["labels"] = labels
        
        return item

print("[OK] Dataset class defined")


[OK] Dataset class defined


In [8]:
# ============================================================================
# AKKADIAN TRANSLATOR CLASS
# ============================================================================

class AkkadianTranslator:
    """
    Main translator class supporting multiple model backends.
    """
    
    def __init__(self, model_path: str = None, device: torch.device = Config.DEVICE):
        self.device = device
        self.model = None
        self.tokenizer = None
        self.model_path = model_path
        
    def load_pretrained_akkadian_model(self, model_name: str = "akkadian-byt5-small-translator"):
        """Load pre-trained Akkadian model from Kaggle Models."""
        
        # Check for Kaggle model paths
        kaggle_model_paths = {
            "akkadian-byt5-small-translator": "/kaggle/input/akkadian-byt5-small-translator/default/1",
            "t5_base_akkadian2english": "/kaggle/input/t5-base-akkadian2english/default/1",
            "T5-Base-Deep-Past": "/kaggle/input/t5-base-deep-past/default/1",
            "Akkadian-T5-Enriched-V2": "/kaggle/input/akkadian-t5-enriched-v2/transformer/1",
            "flan-t5-base": "/kaggle/input/flan-t5/pytorch/base/1",
            "KANISH": "/kaggle/input/kanish/default/1"
        }
        
        model_path = kaggle_model_paths.get(model_name, model_name)
        
        print(f"[LOAD] Loading model: {model_name}")
        print(f"   Path: {model_path}")
        
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_path)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
            self.model.to(self.device)
            self.model.eval()
            print(f"[OK] Model loaded successfully!")
            return True
        except Exception as e:
            print(f"[ERROR] Failed to load model: {e}")
            return False
    
    def load_t5_base(self):
        """Load vanilla T5-base for fine-tuning."""
        print("[LOAD] Loading T5-base model...")
        
        self.tokenizer = T5Tokenizer.from_pretrained("t5-base")
        self.model = T5ForConditionalGeneration.from_pretrained("t5-base")
        self.model.to(self.device)
        
        print(f"[OK] T5-base loaded!")
        print(f"   Parameters: {sum(p.numel() for p in self.model.parameters()):,}")
        
    def translate(
        self, 
        texts: List[str],
        batch_size: int = 8,
        num_beams: int = Config.NUM_BEAMS,
        max_length: int = Config.MAX_TARGET_LENGTH,
        length_penalty: float = Config.LENGTH_PENALTY,
        repetition_penalty: float = Config.REPETITION_PENALTY,
        prefix: str = "translate Akkadian to English: "
    ) -> List[str]:
        """Translate a list of Akkadian texts to English."""
        
        self.model.eval()
        translations = []
        
        for i in tqdm(range(0, len(texts), batch_size), desc="Translating"):
            batch_texts = texts[i:i+batch_size]
            
            # Add prefix
            batch_inputs = [prefix + text for text in batch_texts]
            
            # Tokenize
            inputs = self.tokenizer(
                batch_inputs,
                return_tensors="pt",
                max_length=Config.MAX_SOURCE_LENGTH,
                truncation=True,
                padding=True
            ).to(self.device)
            
            # Generate
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_length=max_length,
                    num_beams=num_beams,
                    length_penalty=length_penalty,
                    repetition_penalty=repetition_penalty,
                    early_stopping=True,
                    no_repeat_ngram_size=3
                )
            
            # Decode
            batch_translations = self.tokenizer.batch_decode(
                outputs, skip_special_tokens=True
            )
            translations.extend(batch_translations)
        
        return translations
    
    def compute_metrics(self, predictions: List[str], references: List[str]) -> Dict:
        """Compute BLEU and chrF++ metrics."""
        
        # BLEU score
        bleu = BLEU()
        bleu_score = bleu.corpus_score(predictions, [references]).score
        
        # chrF++ score
        chrf = CHRF(word_order=2)  # chrF++ includes word n-grams
        chrf_score = chrf.corpus_score(predictions, [references]).score
        
        # Geometric mean (competition metric)
        geometric_mean = np.sqrt(bleu_score * chrf_score)
        
        return {
            "bleu": bleu_score,
            "chrf++": chrf_score,
            "geometric_mean": geometric_mean
        }

print("[OK] AkkadianTranslator class defined")


[OK] AkkadianTranslator class defined


## 5. Model Training & Fine-tuning

Fine-tuning strategy using Hugging Face Trainer with:
- Gradient accumulation for larger effective batch size
- Learning rate warmup
- Early stopping based on validation loss


In [9]:
# ============================================================================
# TRAINING SETUP WITH HUGGING FACE TRAINER
# ============================================================================

# Import training components (lazy import to avoid issues at startup)
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

def prepare_hf_dataset(df: pd.DataFrame, tokenizer, prefix: str = "translate Akkadian to English: "):
    """Prepare dataset in HuggingFace format."""
    
    def preprocess_function(examples):
        inputs = [prefix + text for text in examples["transliteration_clean"]]
        targets = examples["translation_clean"]
        
        model_inputs = tokenizer(
            inputs, 
            max_length=Config.MAX_SOURCE_LENGTH, 
            truncation=True,
            padding="max_length"
        )
        
        labels = tokenizer(
            targets, 
            max_length=Config.MAX_TARGET_LENGTH, 
            truncation=True,
            padding="max_length"
        )
        
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs
    
    # Convert to HuggingFace Dataset
    hf_dataset = HFDataset.from_pandas(df[["transliteration_clean", "translation_clean"]])
    
    # Tokenize
    tokenized_dataset = hf_dataset.map(
        preprocess_function, 
        batched=True,
        remove_columns=hf_dataset.column_names
    )
    
    return tokenized_dataset


def train_model(
    translator: AkkadianTranslator,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame = None,
    output_dir: str = "./akkadian_model",
    num_epochs: int = Config.EPOCHS
):
    """Fine-tune the translation model."""
    
    tokenizer = translator.tokenizer
    model = translator.model
    
    # Prepare datasets
    print("[INFO] Preparing training dataset...")
    train_dataset = prepare_hf_dataset(train_df, tokenizer)
    
    eval_dataset = None
    if val_df is not None:
        print("[INFO] Preparing validation dataset...")
        eval_dataset = prepare_hf_dataset(val_df, tokenizer)
    
    # Data collator
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        label_pad_token_id=-100
    )
    
    # Training arguments
    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch" if eval_dataset else "no",
        learning_rate=Config.LEARNING_RATE,
        per_device_train_batch_size=Config.BATCH_SIZE,
        per_device_eval_batch_size=Config.BATCH_SIZE,
        gradient_accumulation_steps=Config.GRADIENT_ACCUMULATION,
        weight_decay=Config.WEIGHT_DECAY,
        warmup_ratio=Config.WARMUP_RATIO,
        num_train_epochs=num_epochs,
        predict_with_generate=True,
        generation_max_length=Config.MAX_TARGET_LENGTH,
        fp16=torch.cuda.is_available(),
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True if eval_dataset else False,
        metric_for_best_model="eval_loss" if eval_dataset else None,
        greater_is_better=False,
        logging_steps=100,
        report_to="none",
        seed=Config.SEED
    )
    
    # Initialize trainer
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )
    
    # Train
    print("\n[START] Starting training...")
    trainer.train()
    
    # Save final model
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    print(f"\n[OK] Training complete! Model saved to {output_dir}")
    
    return trainer

print("[OK] Training functions defined")


[OK] Training functions defined


## 6. Inference & Submission Generation


In [10]:
# ============================================================================
# INFERENCE PIPELINE
# ============================================================================

def run_inference_pipeline(
    test_df: pd.DataFrame,
    model_name: str = "akkadian-byt5-small-translator",
    use_pretrained: bool = True,
    finetuned_path: str = None
):
    """
    Complete inference pipeline for generating translations.
    """
    
    # Initialize translator
    translator = AkkadianTranslator()
    
    if use_pretrained:
        # Try to load pre-trained Akkadian model
        success = translator.load_pretrained_akkadian_model(model_name)
        
        if not success:
            print("[WARN] Falling back to T5-base...")
            translator.load_t5_base()
    
    elif finetuned_path:
        # Load fine-tuned model
        print(f"[LOAD] Loading fine-tuned model from {finetuned_path}")
        translator.tokenizer = AutoTokenizer.from_pretrained(finetuned_path)
        translator.model = AutoModelForSeq2SeqLM.from_pretrained(finetuned_path)
        translator.model.to(Config.DEVICE)
    
    # Get preprocessed transliterations
    texts = test_df['transliteration_clean'].tolist()
    
    # Generate translations
    print(f"\n[INFO] Generating translations for {len(texts)} samples...")
    translations = translator.translate(texts)
    
    return translations, translator


def create_submission(test_df: pd.DataFrame, translations: List[str], output_path: str = "submission.csv"):
    """Create submission file in required format."""
    
    submission_df = pd.DataFrame({
        'id': test_df['id'],
        'translation': translations
    })
    
    submission_df.to_csv(output_path, index=False)
    print(f"\n[OK] Submission saved to {output_path}")
    print(f"   Shape: {submission_df.shape}")
    
    # Show sample
    print("\nSample predictions:")
    for i in range(min(3, len(submission_df))):
        print(f"   ID {submission_df.iloc[i]['id']}: {submission_df.iloc[i]['translation'][:80]}...")
    
    return submission_df

print("[OK] Inference functions defined")


[OK] Inference functions defined


## 7. Main Execution Pipeline

Choose your strategy below:


In [11]:
# ============================================================================
# MAIN EXECUTION - OPTION 1: USE PRE-TRAINED AKKADIAN MODEL
# ============================================================================

# This is the recommended approach for the competition
# Uses models specifically trained for Akkadian translation

STRATEGY = "pretrained"  # Options: "pretrained", "finetune", "ensemble"

if STRATEGY == "pretrained":
    print("=" * 70)
    print("STRATEGY: Using Pre-trained Akkadian Model")
    print("=" * 70)
    
    # List of available pre-trained models (ordered by priority)
    MODELS_TO_TRY = [
        "akkadian-byt5-small-translator",
        "t5_base_akkadian2english", 
        "T5-Base-Deep-Past",
        "Akkadian-T5-Enriched-V2",
        "KANISH",
        "flan-t5-base"
    ]
    
    # Try models in order
    translator = AkkadianTranslator()
    model_loaded = False
    
    for model_name in MODELS_TO_TRY:
        print(f"\n[TRY] Attempting to load: {model_name}")
        if translator.load_pretrained_akkadian_model(model_name):
            model_loaded = True
            break
    
    if not model_loaded:
        print("\n[WARN] No pre-trained model available. Using T5-base baseline.")
        translator.load_t5_base()

print("\n[OK] Model ready for inference!")


STRATEGY: Using Pre-trained Akkadian Model

[TRY] Attempting to load: akkadian-byt5-small-translator
[LOAD] Loading model: akkadian-byt5-small-translator
   Path: /kaggle/input/akkadian-byt5-small-translator/default/1
[ERROR] Failed to load model: Incorrect path_or_model_id: '/kaggle/input/akkadian-byt5-small-translator/default/1'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

[TRY] Attempting to load: t5_base_akkadian2english
[LOAD] Loading model: t5_base_akkadian2english
   Path: /kaggle/input/t5-base-akkadian2english/default/1
[ERROR] Failed to load model: Incorrect path_or_model_id: '/kaggle/input/t5-base-akkadian2english/default/1'. Please provide either the path to a local folder or the repo_id of a model on the Hub.

[TRY] Attempting to load: T5-Base-Deep-Past
[LOAD] Loading model: T5-Base-Deep-Past
   Path: /kaggle/input/t5-base-deep-past/default/1
[ERROR] Failed to load model: Incorrect path_or_model_id: '/kaggle/input/t5-base-deep-pas

In [12]:
# ============================================================================
# OPTIONAL: FINE-TUNE ON TRAINING DATA
# ============================================================================

# Set to True to fine-tune the model on the competition training data
FINETUNE_MODEL = False

if FINETUNE_MODEL:
    print("=" * 70)
    print("FINE-TUNING MODEL ON COMPETITION DATA")
    print("=" * 70)
    
    # Split training data
    from sklearn.model_selection import train_test_split
    
    train_data, val_data = train_test_split(
        train_df, 
        test_size=0.1, 
        random_state=Config.SEED
    )
    
    print(f"[INFO] Training samples: {len(train_data)}")
    print(f"[INFO] Validation samples: {len(val_data)}")
    
    # Fine-tune
    trainer = train_model(
        translator,
        train_data,
        val_data,
        output_dir="./akkadian_finetuned",
        num_epochs=3
    )
    
    # Validate on held-out data
    val_texts = val_data['transliteration_clean'].tolist()
    val_refs = val_data['translation_clean'].tolist()
    
    print("\nValidation Results:")
    val_preds = translator.translate(val_texts)
    metrics = translator.compute_metrics(val_preds, val_refs)
    
    for metric, value in metrics.items():
        print(f"   {metric}: {value:.4f}")
else:
    print("[SKIP] Skipping fine-tuning (set FINETUNE_MODEL=True to enable)")


[SKIP] Skipping fine-tuning (set FINETUNE_MODEL=True to enable)


In [13]:
# ============================================================================
# GENERATE PREDICTIONS ON TEST SET
# ============================================================================

print("=" * 70)
print("GENERATING PREDICTIONS ON TEST SET")
print("=" * 70)

# Get test transliterations
test_texts = test_df['transliteration_clean'].tolist()

print(f"\n[INFO] Test samples: {len(test_texts)}")
print(f"[INFO] Sample input: {test_texts[0][:100]}...")

# Generate translations
translations = translator.translate(
    test_texts,
    batch_size=8,
    num_beams=5,
    max_length=256
)

print(f"\n[OK] Generated {len(translations)} translations")

# Show sample outputs
print("\nSample Translations:")
print("-" * 70)
for i in range(min(5, len(test_texts))):
    print(f"\n[{i+1}] Akkadian: {test_texts[i][:80]}...")
    print(f"    English:  {translations[i][:80]}...")


GENERATING PREDICTIONS ON TEST SET

[INFO] Test samples: 4
[INFO] Sample input: um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-ilbig_gap da-tim aí-ip-ri-ni kà-ar kà-ar-ma ú wa-bar-ra-tim qí...


Translating: 100%|██████████| 1/1 [00:05<00:00,  5.91s/it]


[OK] Generated 4 translations

Sample Translations:
----------------------------------------------------------------------

[1] Akkadian: um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-ilbig_gap da-tim aí-ip-ri-ni kà-ar kà-ar-m...
    English:  Akkadisch um-ma kà-ru-um ká-ni-ia-m a-na aa-q-ilbig_gap da-tim a-ip-ri­ni kù-ar ...

[2] Akkadian: i-na mup-pì-im aa a-lim ia-tù u„-mì-im a-nim ma-ma-an KÙ AN i-aa-ú-mu-ni i-na né...
    English:  Akkadisch i-na mup-p-im aa a-lim ia-tù u„-m-IM a­nim ma-ma-an K AN i–aa--mu-ni i...

[3] Akkadian: ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na aí-mì-im a-na É GAL-lim i-dí-in l...
    English:  Akkadisch ki-ma mup-p-ni ta-áa-me-a-nich a-mam-kam lu a–na a-m-im a­na É GAL-lim...

[4] Akkadian: me-+e-er mup-pì-ni a-na kà-ar kà-ar-ma ú wa-bar-ra-tim aé-bi„-lá KÙ AN lu a-na D...
    English:  Akkadisch me-+e-er mup-p-ni a-na kà-ar ká-ar-ma  wa-bar-ra-tim aé-bi„-lá K AN lu...


In [14]:
# ============================================================================
# CREATE SUBMISSION FILE
# ============================================================================

print("=" * 70)
print("CREATING SUBMISSION FILE")
print("=" * 70)

# Create submission DataFrame
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'translation': translations
})

# Save to CSV
submission_path = "submission.csv"
submission_df.to_csv(submission_path, index=False)

print(f"\n[OK] Submission file saved: {submission_path}")
print(f"   Shape: {submission_df.shape}")

# Display submission
print("\nSubmission Preview:")
display(submission_df.head(10))

# Verify submission format
print("\nSubmission Verification:")
print(f"   - Columns: {submission_df.columns.tolist()}")
print(f"   - Rows: {len(submission_df)}")
print(f"   - Any missing translations: {submission_df['translation'].isna().sum()}")
print(f"   - Empty translations: {(submission_df['translation'] == '').sum()}")


CREATING SUBMISSION FILE

[OK] Submission file saved: submission.csv
   Shape: (4, 2)

Submission Preview:


,id,translation
0,0,Akkadisch um-ma kà-ru-um ká-ni-ia-m a-na aa-q-...
1,1,Akkadisch i-na mup-p-im aa a-lim ia-tù u„-m-IM...
2,2,Akkadisch ki-ma mup-p-ni ta-áa-me-a-nich a-mam...
3,3,Akkadisch me-+e-er mup-p-ni a-na kà-ar ká-ar-m...



Submission Verification:
   - Columns: ['id', 'translation']
   - Rows: 4
   - Any missing translations: 0
   - Empty translations: 0


## 8. Improvement Strategies

### Immediate Improvements:
1. **Try multiple pre-trained models** and ensemble their predictions
2. **Fine-tune** on competition training data
3. **Data augmentation** using publications.csv OCR text

### Advanced Strategies:
1. **LLM Post-processing**: Use Gemma-3 or Qwen3 to refine translations
2. **Lexicon Integration**: Use eBL dictionary for proper noun handling  
3. **Back-translation**: Generate synthetic training data
4. **Domain Adaptation**: Continue pre-training on published_texts.csv

### Model-Specific Tips:
- **akkadian-byt5**: Good baseline, handles character-level patterns
- **T5-Base-Deep-Past**: Specifically trained for this competition
- **Akkadian T5 Enriched**: Uses augmented training data
- **KANISH**: Novel architecture for low-resource MT

---
**Competition Deadline**: Check Kaggle for current timeline  
**Runtime Limit**: 9 hours (CPU or GPU)


In [15]:
# ============================================================================
# OPTIONAL: ENSEMBLE MULTIPLE MODELS
# ============================================================================

RUN_ENSEMBLE = False  # Set to True to run ensemble

if RUN_ENSEMBLE:
    print("=" * 70)
    print("RUNNING ENSEMBLE OF MULTIPLE MODELS")
    print("=" * 70)
    
    # Models to ensemble
    ensemble_models = [
        "akkadian-byt5-small-translator",
        "T5-Base-Deep-Past",
        "Akkadian-T5-Enriched-V2"
    ]
    
    all_predictions = []
    
    for model_name in ensemble_models:
        print(f"\n[LOAD] Loading: {model_name}")
        
        translator = AkkadianTranslator()
        if translator.load_pretrained_akkadian_model(model_name):
            preds = translator.translate(test_texts, batch_size=8)
            all_predictions.append(preds)
            
            # Free memory
            del translator
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    if len(all_predictions) > 1:
        print(f"\n[OK] Got predictions from {len(all_predictions)} models")
        
        # Simple voting: Use longest translation (often more complete)
        # Alternative: Use BLEU-weighted voting
        ensemble_translations = []
        
        for i in range(len(test_texts)):
            candidates = [preds[i] for preds in all_predictions]
            # Pick the longest translation (heuristic)
            best = max(candidates, key=len)
            ensemble_translations.append(best)
        
        translations = ensemble_translations
        print("[OK] Ensemble complete!")
    else:
        print("[WARN] Only one model loaded, using single model predictions")
else:
    print("[SKIP] Ensemble disabled (set RUN_ENSEMBLE=True to enable)")

print("\n[DONE] Pipeline complete!")


[SKIP] Ensemble disabled (set RUN_ENSEMBLE=True to enable)

[DONE] Pipeline complete!
